# 文档切分器

## 用于将长文档切分为多个短文档，方便后续向量化
## · 语义切分(引入LLM辅助切分)
## · 固定长度切分(按照Token或字符数切分)
## · 递归切分(按照分隔符切分)
## · 滑动窗口切分(固定窗口切分，可与上一chunk有一定重叠率)
## · 结构感知切分(利用文档元数据切分 e.g Markdown or Html)

In [1]:
# 文本
doc = """
城市夜间图书馆观察笔记

城市夜间图书馆并不是一个单纯的阅读场所，它更像是一座在深夜持续运转的公共客厅。晚上十点以后，白天匆忙的人群逐渐散去，留下来的读者往往有着不同的目的：有人准备考试，有人处理工作，有人只是想找一个安静的地方坐一会儿。灯光、座位、书架和低声交谈共同构成了一种特殊的秩序。

一、空间与使用方式
夜间图书馆通常会把空间划分为几个区域。安静阅读区要求手机静音，不建议长时间交谈；协作学习区允许小声讨论，适合小组复习；公共休息区则提供饮水、充电和短暂休息。不同区域之间的边界并不总是靠墙来完成，有时只需要一块地毯、一排书架或一盏亮度不同的灯。值得注意的是，夜间图书馆的核心并不是“延长开放时间”，而是“降低使用门槛”。如果一个读者只需要半小时的安静空间，那么复杂的预约流程和过高的进入条件都会削弱图书馆的公共价值。因此，许多夜间图书馆会保留免预约座位，并在入口处明确标示剩余座位和开放区域。

二、读者行为观察
1. 固定读者往往会选择相同的位置，并把水杯、外套和笔记本摆成熟悉的布局。
2. 临时读者更关注充电口、洗手间和出口位置，通常会在坐下前观察一圈。
3. 考试季的读者倾向于使用协作学习区，但会主动压低音量。
4. 深夜时段的管理重点不是提高声音限制，而是处理占座、饮食和安全隐患。

 三、管理建议
夜间图书馆的管理可以采用“轻规则、强提示”的方式。轻规则意味着不增加过多审批环节，强提示意味着在关键位置清楚说明行为边界。例如，在充电区标注“充满后请让给下一位”，在安静区入口提示“请将手机调至静音”，在休息区提供垃圾分类说明。此外，夜间服务需要建立可追踪的巡检记录。巡检不应只记录“是否正常”，而应记录照明、温度、噪音、占座和设备故障等具体项目。这样既能帮助管理人员发现问题，也能为后续调整开放时间提供依据。
总之，城市夜间图书馆的价值在于为不同节奏的人提供一段可进入、可停留、可离开的公共时间。它不需要成为全天候服务中心，但应当成为城市公共文化设施中一个稳定、低门槛且可预期的夜间节点。
"""

## 1.固定长度切分

### · 字符长度

In [ ]:
from langchain_text_splitters import CharacterTextSplitter, MarkdownHeaderTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=100,  # 块大小
    chunk_overlap=20,  # 重叠字数
    separator=""  # 分隔符
)

# chunks是一个列表，包含所有块，每个块都是一个字符串
chunks = text_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} 字符")

print(f"分为 {len(chunks)} 个块\n")

for i,chunk in enumerate(chunks):
    print(f"----- Chunk {i + 1} ({len(chunk)}个字符) -----")
    print(chunk)

### · Token

In [4]:
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100,  # 块大小
    chunk_overlap=20,  # 重叠字数
    encoding_name="cl100k_base"  # token分词编译器
)

chunks = text_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} 字符")

print(f"分为 {len(chunks)} 个块\n")

for i,chunk in enumerate(chunks):
    print(f"----- Chunk {i + 1} ({len(chunk)}个字符) -----")
    print(chunk)

ReadTimeout: HTTPSConnectionPool(host='openaipublic.blob.core.windows.net', port=443): Read timed out. (read timeout=None)

## 2.递归切分

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 块大小
    chunk_overlap=20,  # 重叠字数
    separators=['\n\n', '，', '。']  # token分词编译器
)

chunks = text_splitter.split_text(doc)

print(f"原始文本长度: {len(doc)} 字符")

print(f"分为 {len(chunks)} 个块\n")

for i,chunk in enumerate(chunks):
    print(f"----- Chunk {i + 1} ({len(chunk)}个字符) -----")
    print(chunk)

原始文本长度: 864 字符
分为 6 个块

----- Chunk 1 (145个字符) -----
城市夜间图书馆观察笔记

城市夜间图书馆并不是一个单纯的阅读场所，它更像是一座在深夜持续运转的公共客厅。晚上十点以后，白天匆忙的人群逐渐散去，留下来的读者往往有着不同的目的：有人准备考试，有人处理工作，有人只是想找一个安静的地方坐一会儿。灯光、座位、书架和低声交谈共同构成了一种特殊的秩序。
----- Chunk 2 (187个字符) -----
一、空间与使用方式
夜间图书馆通常会把空间划分为几个区域。安静阅读区要求手机静音，不建议长时间交谈；协作学习区允许小声讨论，适合小组复习；公共休息区则提供饮水、充电和短暂休息。不同区域之间的边界并不总是靠墙来完成，有时只需要一块地毯、一排书架或一盏亮度不同的灯。
值得注意的是，夜间图书馆的核心并不是“延长开放时间”，而是“降低使用门槛”。如果一个读者只需要半小时的安静空间
----- Chunk 3 (69个字符) -----
，那么复杂的预约流程和过高的进入条件都会削弱图书馆的公共价值。因此，许多夜间图书馆会保留免预约座位，并在入口处明确标示剩余座位和开放区域。
----- Chunk 4 (149个字符) -----
二、读者行为观察
1. 固定读者往往会选择相同的位置，并把水杯、外套和笔记本摆成熟悉的布局。
2. 临时读者更关注充电口、洗手间和出口位置，通常会在坐下前观察一圈。
3. 考试季的读者倾向于使用协作学习区，但会主动压低音量。
4. 深夜时段的管理重点不是提高声音限制，而是处理占座、饮食和安全隐患。
----- Chunk 5 (121个字符) -----
三、管理建议
夜间图书馆的管理可以采用“轻规则、强提示”的方式。轻规则意味着不增加过多审批环节，强提示意味着在关键位置清楚说明行为边界。例如，在充电区标注“充满后请让给下一位”，在安静区入口提示“请将手机调至静音”，在休息区提供垃圾分类说明。
----- Chunk 6 (182个字符) -----
此外，夜间服务需要建立可追踪的巡检记录。巡检不应只记录“是否正常”，而应记录照明、温度、噪音、占座和设备故障等具体项目。这样既能帮助管理人员发现问题，也能为后续调整开放时间提供依据。

总之，城市夜间图书馆的价值在于为不同节奏的人提供一段可进入、可停留、

## 3.结构感知切分

In [10]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


with  open("./resources.md", 'r', encoding="utf-8") as f:
    doc = f.read()

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        # ("#", "H1")参一表示切分依据，参二表示切分后自定义的标题
        ("#", "H1"),
        ("##", "H2"),
        ("###", "H3")
    ],
    strip_headers=True  # 正文是否去除文本中的标题, False:去除
)

# 切分文档，接收str，返回一个list[document]
chunks = markdown_splitter.split_text(doc)

for chunk in chunks:
    print(chunk.model_dump_json(indent=2))  # 将BaseModel转为Json

{
  "id": null,
  "metadata": {
    "H1": "城市夜间图书馆观察笔记"
  },
  "page_content": "城市夜间图书馆并不是一个单纯的阅读场所，它更像是一座在深夜持续运转的公共客厅。晚上十点以后，白天匆忙的人群逐渐散去，留下来的读者往往有着不同的目的：有人准备考试，有人处理工作，有人只是想找一个安静的地方坐一会儿。灯光、座位、书架和低声交谈共同构成了一种特殊的秩序。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "H1": "城市夜间图书馆观察笔记",
    "H2": "一、空间与使用方式"
  },
  "page_content": "夜间图书馆通常会把空间划分为几个区域。安静阅读区要求手机静音，不建议长时间交谈；协作学习区允许小声讨论，适合小组复习；公共休息区则提供饮水、充电和短暂休息。不同区域之间的边界并不总是靠墙来完成，有时只需要一块地毯、一排书架或一盏亮度不同的灯。  \n值得注意的是，夜间图书馆的核心并不是“延长开放时间”，而是“降低使用门槛”。如果一个读者只需要半小时的安静空间，那么复杂的预约流程和过高的进入条件都会削弱图书馆的公共价值。因此，许多夜间图书馆会保留免预约座位，并在入口处明确标示剩余座位和开放区域。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "H1": "城市夜间图书馆观察笔记",
    "H2": "二、读者行为观察"
  },
  "page_content": "1. 固定读者往往会选择相同的位置，并把水杯、外套和笔记本摆成熟悉的布局。\n2. 临时读者更关注充电口、洗手间和出口位置，通常会在坐下前观察一圈。\n3. 考试季的读者倾向于使用协作学习区，但会主动压低音量。\n4. 深夜时段的管理重点不是提高声音限制，而是处理占座、饮食和安全隐患。",
  "type": "Document"
}
{
  "id": null,
  "metadata": {
    "H1": "城市夜间图书馆观察笔记",
    "H2": "三、管理建议"
  },
  "page_content": "夜间图书馆的管理可以采用“